# Практика №9.

## Задание

1. Выбрать фрагмент открытого текста.
2. Сгенерировать гамму шифра требуемой длины одним из генераторов псевдослучайных последовательностей.
3. Зашифровать фрагмент открытого текста.
4. Дешифровать зашифрованный текст.

In [284]:
import math
import random
from scipy.special import gammaincc

In [285]:
def frequency_test(sequence):
    n = len(sequence)
    s = sum([1 if bit == '1' else -1 for bit in sequence])
    s_obs = abs(s) / math.sqrt(n)
    p_value = gammaincc(0.5, s_obs**2 / 2)
    return p_value > 0.01

In [286]:
def block_frequency_test(sequence, block_size=10):
    n = len(sequence)
    num_blocks = n // block_size
    if num_blocks == 0:
        return
    
    proportions = []
    for i in range(num_blocks):
        block = sequence[i * block_size:(i + 1) * block_size]
        proportions.append(block.count('1') / block_size)
    
    chi_squared = 4 * block_size * sum([(p - 0.5) ** 2 for p in proportions])
    p_value = gammaincc(num_blocks / 2, chi_squared / 2)
    return p_value > 0.01

In [287]:
def runs_test(sequence):
    n = len(sequence)
    pi = sequence.count('1') / n
    if abs(pi - 0.5) > (2 / math.sqrt(n)):
        return False
    
    runs = 1
    for i in range(1, n):
        if sequence[i] != sequence[i - 1]:
            runs += 1
            
    expected_runs = 2 * n * pi * (1 - pi)
    variance = 2 * n * pi * (1 - pi) * (1 - 2 * pi)
    z = abs(runs - expected_runs) / math.sqrt(abs(variance))
    p_value = gammaincc(0.5, z**2 / 2)
    return p_value > 0.01

In [288]:
def longest_run_of_ones_test(sequence, block_size=8):
    n = len(sequence)
    num_blocks = n // block_size
    if num_blocks == 0:
        return None
    
    max_run_lengths = []
    for i in range(num_blocks):
        block = sequence[i * block_size:(i + 1) * block_size]
        max_run_lengths.append(max(map(len, block.split('0'))))
    
    observed_max_run = max(max_run_lengths)
    expected_run = 0.5 * block_size
    variance = 0.25 * block_size
    z = (observed_max_run - expected_run) / math.sqrt(variance)
    p_value = gammaincc(0.5, z**2 / 2)
    
    return p_value > 0.01

In [289]:
def serial_test(sequence, pattern_length=2):
    n = len(sequence)
    patterns = {}
    for i in range(n - pattern_length + 1):
        pattern = sequence[i:i + pattern_length]
        patterns[pattern] = patterns.get(pattern, 0) + 1
    
    chi_squared = 0
    expected_count = (n - pattern_length + 1) / (2 ** pattern_length)
    for count in patterns.values():
        chi_squared += (count - expected_count) ** 2 / expected_count
    
    degrees_of_freedom = (2 ** pattern_length) - 1
    p_value = gammaincc(degrees_of_freedom / 2, chi_squared / 2)
    return p_value > 0.01

### 1. Выбрать фрагмент открытого текста.

In [290]:
message = "Lorem ipsum dolor sit amet, consectetur adipiscing elit."

### 2. Сгенерировать гамму шифра требуемой длины одним из генераторов псевдослучайных последовательностей.

In [291]:
def lcg(seed, multiplier=43, increment=23, modulus=2**8, max_count=10000):
    x = seed
    for _ in range(max_count):
        x = (multiplier * x + increment) % modulus
        yield x

### 3. Зашифровать фрагмент открытого текста.

In [292]:
rand = lcg(12798)

In [293]:
encrypted_message = []
for char in message:
    y_1 = next(rand)
    encrypted_message.append(y_1 ^ ord(char))

print("Зашифрованное сообщение:", encrypted_message)

Зашифрованное сообщение: [141, 237, 159, 131, 212, 10, 76, 62, 66, 39, 176, 22, 77, 149, 121, 241, 211, 2, 190, 239, 237, 234, 100, 131, 116, 134, 145, 246, 106, 245, 155, 77, 228, 161, 217, 67, 13, 31, 151, 174, 144, 246, 244, 6, 128, 73, 182, 183, 15, 5, 173, 163, 53, 99, 177, 0]


### 4. Дешифровать зашифрованный текст.

In [294]:
rand = lcg(12798)

In [295]:
decrypted_message = []
for char in encrypted_message:
    y_1 = next(rand)
    decrypted_message.append(y_1 ^ char)

print("Расшифрованное сообщение:", "".join([chr(c) for c in decrypted_message]))

Расшифрованное сообщение: Lorem ipsum dolor sit amet, consectetur adipiscing elit.


---

In [296]:
def collect_binary_sequence(generator, length):
    binary_sequence = ""
    while len(binary_sequence) < length:
        number = next(generator)
        binary_sequence += format(number, '032b')  # 32-битные числа
    return binary_sequence[:length]

In [297]:
rand = lcg(127980121, multiplier=1664525, increment=1013904223, modulus=2**32, max_count=10000)

In [298]:
binary_sequence = collect_binary_sequence(rand, 10000)

print("Frequency Test:", frequency_test(binary_sequence))
print("Block Frequency Test:", block_frequency_test(binary_sequence, block_size=128))
print("Runs Test:", runs_test(binary_sequence))
print("Longest Run of Ones Test:", longest_run_of_ones_test(binary_sequence, block_size=128))
print("Serial Test:", serial_test(binary_sequence, pattern_length=2))

Frequency Test: True
Block Frequency Test: True
Runs Test: True
Longest Run of Ones Test: False
Serial Test: True


In [299]:
def collect_binary_sequence(generator, length):
    binary_sequence = ""
    while len(binary_sequence) < length:
        number = generator()
        binary_sequence += format(number, '032b')  # 32-битные числа
    return binary_sequence[:length]

In [300]:
def random_generator():
    return random.randint(0, 2**32 - 1)

In [301]:
binary_sequence = collect_binary_sequence(random_generator, 10000)

# Запускаем тесты
print("Frequency Test:", frequency_test(binary_sequence))
print("Block Frequency Test:", block_frequency_test(binary_sequence, block_size=128))
print("Runs Test:", runs_test(binary_sequence))
print("Longest Run of Ones Test:", longest_run_of_ones_test(binary_sequence, block_size=128))
print("Serial Test:", serial_test(binary_sequence, pattern_length=2))

Frequency Test: True
Block Frequency Test: True
Runs Test: False
Longest Run of Ones Test: False
Serial Test: True
